## 上涨家数

In [ ]:
# 获取所有指数列表
indexes = get_all_securities(types=['index'], date=None)

In [ ]:
# 找到上证指数的代码
indexes[indexes.display_name.str.contains("上证指数")]

In [ ]:
# 获取指定日期上证指数成分股列表
codes= get_index_stocks('000001.XSHG', date='2026-08-27')

In [ ]:
len(codes)

In [ ]:
codes[:10]

In [ ]:
from datetime import datetime
today = datetime.date(datetime.now())

In [ ]:
# 获取指定指定日期的量价信息
df = get_price(
    codes, 
    start_date=None,
    end_date=today.strftime('%Y-%m-%d'), 
    frequency='daily', 
    fields=None, 
    skip_paused=False, 
    fq='pre',
    count=2, 
    panel=False, 
    fill_paused=True)

In [ ]:
df.head()

In [ ]:
type(df),df.columns

In [ ]:
print(df.head())

In [ ]:
import pandas as pd
# 长表 -> 宽表:每行一只股票,每列一个交易日
close_wide = df.pivot(index='code', columns='time', values='close')

In [ ]:
close_wide.head()

In [ ]:
# 排除没有任何成交的股票(整列都是 NaN)
close_wide = close_wide.dropna(how='any')

In [ ]:
# 确保列按日期升序(昨天在前、今天在后)
close_wide = close_wide[sorted(close_wide.columns)]
yesterday, today = close_wide.columns

In [ ]:
yesterday,today

In [ ]:
# 严格上涨:今日 close > 昨日 close
up_count = (close_wide[today] > close_wide[yesterday]).sum()

In [ ]:
# 顺便把下跌、平盘也算出来
down_count = (close_wide[today] < close_wide[yesterday]).sum()
flat_count = (close_wide[today] == close_wide[yesterday]).sum()

In [ ]:
print(f"上涨: {up_count}  下跌: {down_count}  平盘: {flat_count}")

In [ ]:
1283+838+66

In [ ]:
from jqdata import get_trade_days

# 往前推 3 个月的交易日,JDATA 是一支一支排好序的
today = pd.Timestamp.today().strftime('%Y-%m-%d')
dates = get_trade_days(end_date=today, count=63)  # 约 3 个月的交易日数
# 想要固定窗口(比如 2026-05-27 ~ 今天):
# dates = get_trade_days(start_date='2026-05-27', end_date=today)

In [ ]:
dates

## 提取函数

好嘞，先把思路捋清楚，再聊实现。

## 上涨家数的算法描述

**核心定义**
对于上证指数成分股（实际就是上交所 A 股），逐只比较「今日收盘价」与「昨日收盘价」，收盘价严格高于昨日收盘价的股票数量，即为「上涨家数」。

**算法步骤**

1. **确定成分股范围**
   - 上证指数的成分股 = 上交所全部 A 股（沪市主板、科创板）
   - 通常剔除 B 股、退市股、停牌超过一定时长的股

2. **数据准备**
   - 取每只成分股今日和昨日的收盘价
   - 标记「是否有效」：
     - 停牌：今日或昨日收盘价为空 → 剔除
     - 新股：上市日期晚于昨日 → 没有「昨日收盘价」 → 剔除
     - 一字涨跌停：如果需要单独统计涨跌停家数，可再标记

3. **逐只比较**
   - 对每只有效股票：判定 `close_today > close_yesterday`
   - 三分类：上涨 / 下跌 / 平盘（close 相等）
   - 平盘通常**不计入**上涨或下跌

4. **汇总**
   - `上涨家数 = 满足 close_today > close_yesterday 的股票数量`
   - 同步可以得到：下跌家数、平盘家数、停牌家数

5. **可选：加权与不加权**
   - 简单家数：直接计数（题目要的就是这个）
   - 如果要算「贡献度」：按市值或自由流通市值加权
   - 还可以衍生：涨跌幅中位数、平均涨跌幅、涨停家数、跌停家数

**几个常见坑**

- 停牌 / 复牌当日的处理：复牌首日虽然有「昨日收盘价」但可能因长期停牌不可比，行业里通常**剔除**或**单独标注**
- ST 股：一般**保留**在统计里（但和普通股涨跌停幅度不同）
- 收盘价字段的选择：有的行情源给的是「最新价」而不是「收盘价」，盘后取数要用真正的 close 字段
- 数据对齐：今日 / 昨日必须是相邻的两个交易日（注意节假日、半天交易日）

如果你想顺手把**下跌家数、平盘家数、涨停家数、跌停家数**一起算出来，算法框架基本不变，只是判定条件多了几个分支。需要我后面写代码的时候说一声。

对了，你平时主要是在做策略研究、还是行情监控/看板这类？这俩场景对「上涨家数」的口径会差挺多的。

### 策略研究和行情监控/看板对上涨家数口径的不同

核心区别:你要的是「信号」还是「快照」

| 维度 | 策略研究 | 行情监控/看板 |
|---|---|---|
| **目的** | 当成模型特征 / 择时因子 | 当下市场冷热快照 |
| **时间视角** | 历史可比(2010 vs 2024 都能用) | 关注"现在" |
| **数值取向** | 比例、变化率、Z-Score | 绝对家数、涨停/跌停家数 |
| **口径敏感度** | 极敏感(动一个数,回测全乱) | 容忍(差几十家无所谓) |
| **样本范围** | 严格定义(成分股/全 A、固定停牌处理) | 灵活(全 A 也行,沪市也行) |
| **核心衍生** | 宽度指数、N 日均线、占比的滚动分位 | 涨跌停、连板、炸板率 |
| **数据量级** | 需要 5~10 年以上稳定历史 | 当日 + 最近 N 日即可 |

具体举个例子

**同一个"上涨家数 = 2500 家"**,两个场景的解读完全不一样:

- **策略视角**:「今天上涨 2500,总样本 5000,占比 50%,过去 20 日均值 45%,Z-Score = +1.2,接近超买阈值」→ 触发信号
  - 必须保证 2500 这个数字是**历史可比的**,否则回测全错
  - 必须固定成分股、固定停牌处理、明确是否算 ST

- **监控视角**:「上涨 2500,下跌 1800,涨停 47,跌停 12,炸板 8 家」→ 一眼看出盘面强度
  - 当下口径算对就行,历史可比性不重要
  - 经常还要拆行业板块、北上资金、两市成交额一起摆
  
你前面代码其实偏监控风格

`get_index_stocks` 当日 + 前一日 close + 算 up/down,这套路典型是**当日快照**的玩法,不是回测用。

如果后面要往策略走,得改:
- 用 `get_all_securities` + 过滤条件**锁定一个固定股票池**(比如全 A、剔除 ST、剔除上市 < 60 日的)
- 停牌一律剔除(不要 `fill_paused=True`)
- 数据一次性拉长周期(比如 5 年),口径**全程一致**
- 算 up_ratio 而不是绝对家数,再做滚动均值/分位数

In [ ]:
import pandas as pd
from jqdata import get_trade_days

def get_daily_breadth(start_date, end_date, index_code='000001.XSHG'):
    """
    计算指定日期范围内,每天的指数成分股涨跌家数
    
    Parameters
    ----------
    start_date : str, 例如 '2026-06-01'
    end_date   : str, 例如 '2026-08-27'
    index_code : str, 默认上证指数 '000001.XSHG'
    
    Returns
    -------
    pd.DataFrame: columns = ['date', 'up', 'down', 'flat', 'total']
    """
    trade_days = get_trade_days(start_date=start_date, end_date=end_date)
    
    rows = []
    for d in trade_days:
        d_str = pd.Timestamp(d).strftime('%Y-%m-%d')
        
        # 1) 当日成分股
        codes = get_index_stocks(index_code, date=d_str)
        if not codes:
            continue
        
        # 2) 当日 + 前一交易日的量价
        df = get_price(
            codes,
            end_date=d_str,
            frequency='daily',
            skip_paused=False,
            fq='pre',
            count=2,
            panel=False,
            fill_paused=True,
        )
        if df.empty:
            continue
        
        # 3) 长表 -> 宽表 (一行一只股票,一列一个交易日)
        close_wide = df.pivot(index='code', columns='time', values='close')
        close_wide = close_wide.dropna(how='any')          # 排除新股/无前一交易日
        if close_wide.shape[1] < 2:
            continue
        
        close_wide = close_wide[sorted(close_wide.columns)]
        prev_day, today = close_wide.columns
        
        up   = (close_wide[today] >  close_wide[prev_day]).sum()
        down = (close_wide[today] <  close_wide[prev_day]).sum()
        flat = (close_wide[today] == close_wide[prev_day]).sum()
        
        rows.append({
            'date':  d_str,
            'up':    int(up),
            'down':  int(down),
            'flat':  int(flat),
            'total': int(up + down + flat),
        })
    
    return pd.DataFrame(rows)

几个需要注意的点:

- **每次循环都调一次 `get_index_stocks` 和 `get_price`**,因为成分股是按日变化的(有调样、纳入/剔除),不能复用;如果跑长周期(比如 1 年+),想优化的话可以把成分股按日缓存,价格一次性拉 [start-1, end] 再切片。
- **`fill_paused=True` 会把停牌日的 close 填成上一交易日**,所以停牌股会被算进 `flat`(平盘)。如果你想严格剔除停牌股,把 `fill_paused=False`,然后在 `pivot` 后用 `dropna(how='any')` 自然过滤。
- **新股**:上市首日没有「昨日收盘价`,`get_price(count=2)` 实际只能拿到 1 行,pivot 后该股票被 `dropna(how='any')` 自动剔除,所以 `total` 严格说是「当日有可比收盘价的家数」,不是成分股总数。
- **stops 暂未统计**(涨停/跌停家数),后面要的话告诉我,加几行就行。

In [ ]:
# 最近 3 个月的上证成分股每日涨跌家数
today = pd.Timestamp.today()
end   = today.strftime('%Y-%m-%d')
start = (today - pd.DateOffset(months=3)).strftime('%Y-%m-%d')
breadth = get_daily_breadth(start, end)
print(breadth.tail())

In [ ]:
breadth = breadth.set_index('date')
breadth.index = pd.to_datetime(breadth.index)

镜像图

In [ ]:
# ================================================================
# 3. 可视化(等距横轴 + 三张图)
# ================================================================
x         = np.arange(len(breadth))                    # 均匀横轴
step      = max(1, len(breadth) // 10)
positions = np.arange(0, len(breadth), step)
labels    = [breadth.index[i].strftime('%m-%d') for i in positions]
def set_uniform_x(ax):
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_xlim(x[0], x[-1])
    ax.set_xlabel('交易日')

In [ ]:
# ---------- 图 1:镜像图(涨跌家数) ----------
fig, ax = plt.subplots(figsize=(14, 6))
up   = breadth['up'].to_numpy()
down = breadth['down'].to_numpy()
ax.fill_between(x,  up,   0, color='red',   alpha=0.7, label='上涨')
ax.fill_between(x, -down, 0, color='green', alpha=0.7, label='下跌')
ax.axhline(0, color='black', linewidth=0.8)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{abs(int(v))}'))
ax.text(0.01, 0.95, '↑ 上涨', transform=ax.transAxes, color='red',   fontsize=12, va='top')
ax.text(0.01, 0.05, '↓ 下跌', transform=ax.transAxes, color='green', fontsize=12, va='bottom')
set_uniform_x(ax)
ax.set_title('上证指数 每日涨跌家数')
ax.set_ylabel('家数(取绝对值)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
breadth['up_ratio']   = breadth['up']   / breadth['total']
breadth['down_ratio'] = breadth['down'] / breadth['total']
breadth['flat_ratio'] = breadth['flat'] / breadth['total']
breadth['net_ratio']  = (breadth['up'] - breadth['down']) / breadth['total']

import matplotlib.pyplot as plt

plt.rcParams['axes.unicode_minus'] = False
    
# ---------- 图 2:净宽度(单线,推荐) ----------
fig, ax = plt.subplots(figsize=(14, 4))
net = breadth['net_ratio'].to_numpy()
ax.set_ylim(-1, 1)

net_ma5 = breadth['net_ratio'].rolling(5).mean().to_numpy()
ax.plot(x, net_ma5, color='black', linewidth=1.2, label='5日均线')

ax.axhline( 0.5, color='red',   linestyle=':', linewidth=0.8, alpha=0.6)
ax.axhline(-0.5, color='green', linestyle=':', linewidth=0.8, alpha=0.6)

ax.plot(x, net, color='#c0392b', linewidth=1.2, label='净上涨占比')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.fill_between(x, net, 0, where=net > 0, color='red',   alpha=0.4, interpolate=True)
ax.fill_between(x, net, 0, where=net < 0, color='green', alpha=0.4, interpolate=True)
set_uniform_x(ax)
ax.set_title('上证指数 每日净宽度(上涨占比 - 下跌占比)')
ax.set_ylabel('占比')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- 图 3:堆叠面积(看占比结构) ----------
fig, ax = plt.subplots(figsize=(14, 5))
ax.stackplot(
    x,
    breadth['up_ratio'].to_numpy(),
    breadth['flat_ratio'].to_numpy(),
    breadth['down_ratio'].to_numpy(),
    labels=['上涨', '平盘', '下跌'],
    colors=['#e74c3c', '#bdc3c7', '#27ae60'],
    alpha=0.85,
)
ax.axhline(0.5, color='white', linestyle='--', linewidth=0.8)
set_uniform_x(ax)
ax.set_title('上证指数 每日涨跌占比(堆叠)')
ax.set_ylabel('占比')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

跑完应该看到三张图,横轴完全等距,数据没问题的情况下:
- **图 1**: 红色=上涨家数,绿色=下跌家数,镜像式展示,带文字标注
- **图 2**: 红色单线,市场偏强时往上、偏弱时往下,**最推荐看板用**
- **图 3**: 三段堆叠,中间灰色"平盘"带把视觉镜像感打散